In [38]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os, sys

In [39]:
import pandas as pd

df = pd.read_csv("updated_df.csv")
df_varian = df[df["MachineType"] == "TrueBeam"]
df_elekta = df[df["MachineType"] == "Agility"]


In [40]:
X = df[['Area', 'Circumference', 'MU', "Span" ,'CoA','MCS','Mu_per_dpf']]
y = df["AD"]

In [41]:
X_varian = df_varian[['Area', 'Circumference', 'MU', "Span" ,'CoA','MCS','Mu_per_dpf']]
y_varian = df_varian["AD"]

In [42]:
X_elekta = df_elekta[['Area', 'Circumference', 'MU', "Span" ,'CoA','MCS','Mu_per_dpf']]
y_elekta = df_elekta["AD"]

In [43]:

# # scale X
scale = StandardScaler()
scaled_X = scale.fit_transform(X)

# # train test split
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(scaled_X, y, test_size = 0.25, random_state = 4)


In [44]:
scale = StandardScaler()
scaled_X_e = scale.fit_transform(X_elekta)
X_train_E, X_test_E, y_train_E, y_test_E = train_test_split(X_elekta, y_elekta, test_size=0.25, random_state=42)

In [45]:
scale = StandardScaler()
scaled_X_v = scale.fit_transform(X_varian)
X_train_V, X_test_V, y_train_V, y_test_V = train_test_split(X_varian, y_varian, test_size=0.25, random_state=42)

RandomForestRegressor(max_depth=9, n_estimators=10)

In [46]:


from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# Each dataset: (X_train, X_test, y_train, y_test)
datasets = {
    "Full":   (X_train_full, X_test_full, y_train_full, y_test_full),
    "Elekta": (X_train_E, X_test_E, y_train_E, y_test_E),
    "Varian": (X_train_V, X_test_V, y_train_V, y_test_V)
}

param_grid = {
    'n_estimators': [5, 10, 15, 20],
    'max_depth': [2, 5, 7, 9]
}

results = []

for name, (X_train, X_test, y_train, y_test) in datasets.items():
    print(f"\n=== Running Random Forest for {name} dataset ===")

    # Grid search for hyperparameter tuning
    grid = GridSearchCV(
        RandomForestRegressor(random_state=42),
        param_grid=param_grid,
        cv=10,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_n = grid.best_params_["n_estimators"]
    best_depth = grid.best_params_["max_depth"]
    print(f"Best params for {name}: n_estimators={best_n}, max_depth={best_depth}")

    # Fit final model
    final = RandomForestRegressor(
        n_estimators=best_n,
        max_depth=best_depth,
        random_state=42
    )
    final.fit(X_train, y_train)

    # Predict and evaluate
    y_pred = final.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"Performance on {name} test set:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE : {mae:.4f}")
    print(f"  R²  : {r2:.4f}")

    # Store results
    results.append({
        "Dataset": name,
        "Best_n_estimators": best_n,
        "Best_max_depth": best_depth,
        "Best_CV_Score": grid.best_score_,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

# Summary table
results_df = pd.DataFrame(results)
print("\n=== Summary of Model Results ===")
print(results_df.round(4))



=== Running Random Forest for Full dataset ===
Best params for Full: n_estimators=20, max_depth=9
Performance on Full test set:
  RMSE: 5.2063
  MAE : 4.0391
  R²  : 0.5316

=== Running Random Forest for Elekta dataset ===
Best params for Elekta: n_estimators=20, max_depth=9
Performance on Elekta test set:
  RMSE: 5.1108
  MAE : 3.7032
  R²  : 0.5339

=== Running Random Forest for Varian dataset ===
Best params for Varian: n_estimators=20, max_depth=9
Performance on Varian test set:
  RMSE: 4.5046
  MAE : 3.2991
  R²  : 0.5250

=== Summary of Model Results ===
  Dataset  Best_n_estimators  Best_max_depth  Best_CV_Score    RMSE     MAE  \
0    Full                 20               9         0.4790  5.2063  4.0391   
1  Elekta                 20               9         0.4789  5.1108  3.7032   
2  Varian                 20               9         0.2900  4.5046  3.2991   

       R2  
0  0.5316  
1  0.5339  
2  0.5250  


In [47]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd


# Parameter grid for tuning
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1]
}

results = []

for name, (X_train, X_test, y_train, y_test) in datasets.items():
    print(f"\n=== Running XGBoost for {name} dataset ===")

    # Define base model
    xgbr = xgb.XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    # Grid search with 10-fold CV
    grid = GridSearchCV(
        xgbr,
        param_grid=param_grid,
        cv=10,
        scoring="r2",   # use R² as CV metric
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    # Best parameters
    best_params = grid.best_params_
    print(f"Best params for {name}: {best_params}")

    # Fit final model with best params
    best_xgbr = xgb.XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=42
    )
    best_xgbr.fit(X_train, y_train)

    # Predict on test set
    y_pred = best_xgbr.predict(X_test)

    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"Performance on {name} test set:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE : {mae:.4f}")
    print(f"  R²  : {r2:.4f}")

    # Store results
    results.append({
        "Dataset": name,
        "Best_n_estimators": best_params["n_estimators"],
        "Best_max_depth": best_params["max_depth"],
        "Best_learning_rate": best_params["learning_rate"],
        "Best_CV_R2": grid.best_score_,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

# Summary table
results_df = pd.DataFrame(results)
print("\n=== XGBoost Results Summary ===")
print(results_df.round(4))



=== Running XGBoost for Full dataset ===
Best params for Full: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200}
Performance on Full test set:
  RMSE: 4.9758
  MAE : 3.8696
  R²  : 0.5722

=== Running XGBoost for Elekta dataset ===
Best params for Elekta: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Performance on Elekta test set:
  RMSE: 4.9826
  MAE : 3.6679
  R²  : 0.5570

=== Running XGBoost for Varian dataset ===
Best params for Varian: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}
Performance on Varian test set:
  RMSE: 4.7143
  MAE : 3.3873
  R²  : 0.4797

=== XGBoost Results Summary ===
  Dataset  Best_n_estimators  Best_max_depth  Best_learning_rate  Best_CV_R2  \
0    Full                200               6                0.05      0.5053   
1  Elekta                200               5                0.10      0.4958   
2  Varian                200               3                0.05      0.3109   

     RMSE     MAE      R2  
0  4.97

In [48]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

datasets = {
    "Full":   (X_train_full, X_test_full, y_train_full, y_test_full),
    "Elekta": (X_train_E, X_test_E, y_train_E, y_test_E),
    "Varian": (X_train_V, X_test_V, y_train_V, y_test_V)
}

param_grid = {
    "max_depth": [2, 3, 4, 5, 6, 8, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

results = []

for name, (X_train, X_test, y_train, y_test) in datasets.items():
    print(f"\n=== Decision Tree for {name} dataset ===")
    grid = GridSearchCV(
        DecisionTreeRegressor(random_state=42),
        param_grid=param_grid,
        cv=10,
        scoring="r2",
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    best_params = grid.best_params_
    print(f"Best params for {name}: {best_params}")
    tree = DecisionTreeRegressor(**best_params, random_state=42)
    tree.fit(X_train, y_train)
    y_pred = tree.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"Performance on {name} test set:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE : {mae:.4f}")
    print(f"  R²  : {r2:.4f}")
    results.append({
        "Dataset": name,
        "Best_max_depth": best_params["max_depth"],
        "Best_min_samples_split": best_params["min_samples_split"],
        "Best_min_samples_leaf": best_params["min_samples_leaf"],
        "Best_CV_R2": grid.best_score_,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

results_df = pd.DataFrame(results)
print("\n=== Decision Tree Results Summary ===")
print(results_df.round(4))



=== Decision Tree for Full dataset ===
Best params for Full: {'max_depth': 6, 'min_samples_leaf': 4, 'min_samples_split': 2}
Performance on Full test set:
  RMSE: 5.9226
  MAE : 4.5131
  R²  : 0.3938

=== Decision Tree for Elekta dataset ===
Best params for Elekta: {'max_depth': 4, 'min_samples_leaf': 1, 'min_samples_split': 2}
Performance on Elekta test set:
  RMSE: 6.1086
  MAE : 4.4221
  R²  : 0.3341

=== Decision Tree for Varian dataset ===
Best params for Varian: {'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2}
Performance on Varian test set:
  RMSE: 5.7623
  MAE : 3.9809
  R²  : 0.2227

=== Decision Tree Results Summary ===
  Dataset  Best_max_depth  Best_min_samples_split  Best_min_samples_leaf  \
0    Full               6                       2                      4   
1  Elekta               4                       2                      1   
2  Varian               2                       2                      1   

   Best_CV_R2    RMSE     MAE      R2  
0

In [49]:
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

datasets = {
    "Full":   (X_train_full, X_test_full, y_train_full, y_test_full),
    "Elekta": (X_train_E, X_test_E, y_train_E, y_test_E),
    "Varian": (X_train_V, X_test_V, y_train_V, y_test_V)
}

kernels = ['linear', 'poly', 'rbf']
results = []

for name, (X_train, X_test, y_train, y_test) in datasets.items():
    print(f"\n=== SVM Regression for {name} dataset ===")
    for kernel in kernels:
        print(f"\nKernel: {kernel}")
        svr_model = make_pipeline(
            StandardScaler(),
            SVR(kernel=kernel, C=100, gamma='auto', epsilon=0.5)
        )
        svr_model.fit(X_train, y_train)
        y_pred = svr_model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE : {mae:.4f}")
        print(f"R²  : {r2:.4f}")
        results.append({
            "Dataset": name,
            "Kernel": kernel,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2
        })

results_df = pd.DataFrame(results)
print("\n=== SVM Regression Results Summary ===")
print(results_df.round(4))



=== SVM Regression for Full dataset ===

Kernel: linear
RMSE: 6.3155
MAE : 4.5673
R²  : 0.3107

Kernel: poly
RMSE: 6.8713
MAE : 4.8930
R²  : 0.1841

Kernel: rbf
RMSE: 5.5440
MAE : 4.0771
R²  : 0.4689

=== SVM Regression for Elekta dataset ===

Kernel: linear
RMSE: 6.2794
MAE : 4.3448
R²  : 0.2963

Kernel: poly
RMSE: 6.1459
MAE : 4.4870
R²  : 0.3259

Kernel: rbf
RMSE: 5.5584
MAE : 3.8334
R²  : 0.4486

=== SVM Regression for Varian dataset ===

Kernel: linear
RMSE: 6.2045
MAE : 3.8445
R²  : 0.0988

Kernel: poly
RMSE: 5.6773
MAE : 3.4824
R²  : 0.2455

Kernel: rbf
RMSE: 5.2730
MAE : 3.4698
R²  : 0.3491

=== SVM Regression Results Summary ===
  Dataset  Kernel    RMSE     MAE      R2
0    Full  linear  6.3155  4.5673  0.3107
1    Full    poly  6.8713  4.8930  0.1841
2    Full     rbf  5.5440  4.0771  0.4689
3  Elekta  linear  6.2794  4.3448  0.2963
4  Elekta    poly  6.1459  4.4870  0.3259
5  Elekta     rbf  5.5584  3.8334  0.4486
6  Varian  linear  6.2045  3.8445  0.0988
7  Varian    poly

In [50]:
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

def logit(p):
    return np.log(p / (1 - p))

def inv_logit(z):
    return 1 / (1 + np.exp(-z))

datasets = {
    "Full":   (X_train_full, X_test_full, y_train_full, y_test_full),
    "Elekta": (X_train_E, X_test_E, y_train_E, y_test_E),
    "Varian": (X_train_V, X_test_V, y_train_V, y_test_V)
}

models = {
    "Lasso": Lasso(alpha=0.1, max_iter=10000),
    "Ridge": Ridge(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
}

results = []

for name, (X_train, X_test, y_train, y_test) in datasets.items():
    print(f"\n=== Logit Regression for {name} dataset ===")
    
    eps = 1e-6
    y_train_s = np.clip(y_train / 100, eps, 1 - eps)
    y_test_s = np.clip(y_test / 100, eps, 1 - eps)
    y_train_logit = logit(y_train_s)
    
    for model_name, model in models.items():
        print(f"\nModel: {model_name}")
        pipe = make_pipeline(StandardScaler(), model)
        pipe.fit(X_train, y_train_logit)
        y_pred_logit = pipe.predict(X_test)
        y_pred = inv_logit(y_pred_logit) * 100
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE : {mae:.4f}")
        print(f"R²  : {r2:.4f}")
        results.append({
            "Dataset": name,
            "Model": model_name,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2
        })

results_df = pd.DataFrame(results)
print("\n=== Logit-Transformed Regression Results Summary ===")
print(results_df.round(4))



=== Logit Regression for Full dataset ===

Model: Lasso
RMSE: 6.6287
MAE : 4.7118
R²  : 0.2407

Model: Ridge
RMSE: 6.1645
MAE : 4.4952
R²  : 0.3433

Model: ElasticNet
RMSE: 6.4593
MAE : 4.6244
R²  : 0.2790

=== Logit Regression for Elekta dataset ===

Model: Lasso
RMSE: 6.4967
MAE : 4.4552
R²  : 0.2468

Model: Ridge
RMSE: 6.0239
MAE : 4.2012
R²  : 0.3524

Model: ElasticNet
RMSE: 6.3096
MAE : 4.3475
R²  : 0.2896

=== Logit Regression for Varian dataset ===

Model: Lasso
RMSE: 6.4602
MAE : 3.8816
R²  : 0.0230

Model: Ridge
RMSE: 6.5231
MAE : 3.9576
R²  : 0.0039

Model: ElasticNet
RMSE: 6.4180
MAE : 3.8491
R²  : 0.0357

=== Logit-Transformed Regression Results Summary ===
  Dataset       Model    RMSE     MAE      R2
0    Full       Lasso  6.6287  4.7118  0.2407
1    Full       Ridge  6.1645  4.4952  0.3433
2    Full  ElasticNet  6.4593  4.6244  0.2790
3  Elekta       Lasso  6.4967  4.4552  0.2468
4  Elekta       Ridge  6.0239  4.2012  0.3524
5  Elekta  ElasticNet  6.3096  4.3475  0.2896